# 工具箱

上一章，学习助手有了两个工具。回头看看 `add_days` 是怎么定义的：先写一个函数，再写一大段 JSON Schema，告诉大模型这个函数叫什么、有哪些参数、每个参数是什么类型。

仔细想想，这段说明里的大部分信息，函数自己就已经有了：

| JSON Schema 里写的 | 函数里本来就有的 |
|---|---|
| 工具的名字 `add_days` | 函数名 |
| 有两个参数 `date` 和 `days` | 参数列表 |
| `date` 是字符串，`days` 是整数 | 类型标注 `date: str`、`days: int` |
| 两个参数都是必填的 | 参数有没有默认值 |
| 工具的用途 | 函数的文档字符串（docstring） |

同样的信息写两遍，不光麻烦，还容易出错：哪天改了函数，忘了改说明，大模型拿到的就是一份过期的说明书。

这一章，我们让程序**直接从函数里读出这些信息，自动生成说明**。这样，函数本身就是唯一的说明书。

In [ ]:
import inspect
import json
import os
import uuid
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
from enum import Enum
from pathlib import Path
from typing import Annotated, Callable, List, get_args, get_origin, get_type_hints
from zoneinfo import ZoneInfo

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam, ToolParam, TextBlockParam, ToolUseBlockParam, ToolResultBlockParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam
from openai.types.chat import ChatCompletionToolParam, ChatCompletionMessageToolCallParam, ChatCompletionToolMessageParam

In [ ]:
def note(label: str, text: str):
    """Print one step of the agent's work, indented, so it stands apart from questions and answers."""
    lines = str(text).splitlines() or [""]
    print(f"  * {label}: {lines[0]}")
    for line in lines[1:]:
        print(f"    {line}")

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"
    TOOL = "Tool"

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [ ]:
class ToolCall:

    def __init__(self, id: str, name: str, arguments: dict):
        self.id = id
        self.name = name
        self.arguments = arguments

    def to_dict(self) -> dict:
        return {"id": self.id, "name": self.name, "arguments": self.arguments}

    def __repr__(self) -> str:
        return f"{self.name}({json.dumps(self.arguments, ensure_ascii=False)})"

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str, tool_calls: List[ToolCall] = None):
        super().__init__(MsgType.ASSISTANT)
        self.content = content
        self.tool_calls = tool_calls or []

    def summary(self) -> str:
        if self.tool_calls:
            return f"{self.content} [call: {self.tool_calls}]"
        return self.content

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.content,
                "tool_calls": [call.to_dict() for call in self.tool_calls]}

In [ ]:
class ToolResultMessage(Message):

    def __init__(self, tool_call_id: str, name: str, result: str):
        super().__init__(MsgType.TOOL)
        self.tool_call_id = tool_call_id
        self.name = name
        self.result = result

    def summary(self) -> str:
        return f"{self.name} -> {self.result}"

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "tool_call_id": self.tool_call_id,
                "name": self.name, "content": self.result}

In [ ]:
def message_from_dict(data: dict) -> Message:
    if data["type"] == MsgType.ASSISTANT.value:
        return AssistantMessage(data["content"], [ToolCall(**c) for c in data.get("tool_calls", [])])
    if data["type"] == MsgType.TOOL.value:
        return ToolResultMessage(data["tool_call_id"], data["name"], data["content"])
    classes = {MsgType.SYSTEM.value: SystemMessage, MsgType.USER.value: UserMessage}
    return classes[data["type"]](data["content"])

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

## Tool 类的小改动

`Tool` 只多了一个 `__call__` 方法。有了它，`Tool` 对象可以像函数一样直接调用，比如 `get_current_time("Asia/Tokyo")`。为什么需要它，等一下看到装饰器就明白了。

In [ ]:
class Tool:

    def __init__(self, name: str, description: str, parameters: dict, function: Callable):
        self.name = name
        self.description = description
        self.parameters = parameters  # JSON Schema of the arguments
        self.function = function

    def run(self, arguments: dict) -> str:
        return str(self.function(**arguments))

    def __call__(self, *args, **kwargs):
        return self.function(*args, **kwargs)

## 用装饰器把函数变成工具

我们要写一个函数 `tool`：传入一个普通的 Python 函数，返回一个 `Tool` 对象，名字、描述、参数说明全部自动生成。它用到了几个 Python 的知识点：

**类型标注和 `Annotated`**

函数参数后面的 `: str`、`: int` 叫作**类型标注**，说明这个参数应该是什么类型。但参数的**说明文字**放在哪里呢？Python 提供了 `Annotated`，可以给类型附带额外的信息：

```python
date: Annotated[str, "Start date in YYYY-MM-DD format"]
```

意思是：`date` 的类型是 `str`，另外附带一段说明文字。

**`inspect`：读取函数的信息**

Python 自带的 `inspect` 模块，可以在程序运行时查看一个函数的各种信息：

- `inspect.signature(function)`：参数列表，包括每个参数的名字和默认值；
- `inspect.getdoc(function)`：函数的文档字符串；
- `get_type_hints(function, include_extras=True)`：每个参数的类型标注（`include_extras=True` 表示把 `Annotated` 附带的说明也保留下来）。

`tool` 函数做的事情就是：遍历每个参数，查出它的类型，用 `JSON_TYPES` 这张对照表把 Python 的类型换成 JSON Schema 的类型名；如果有 `Annotated` 说明，就取出来作为参数说明；没有默认值的参数，就是必填的。

In [ ]:
JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}


def tool(function: Callable) -> Tool:
    """Decorator: build a Tool from a function's name, docstring and type hints."""
    hints = get_type_hints(function, include_extras=True)
    properties, required = {}, []

    for name, param in inspect.signature(function).parameters.items():
        hint, description = hints.get(name, str), None
        if get_origin(hint) is Annotated:  # Annotated[int, "description"]
            hint, description = get_args(hint)[:2]
        properties[name] = {"type": JSON_TYPES.get(hint, "string")}
        if description:
            properties[name]["description"] = description
        if param.default is inspect.Parameter.empty:
            required.append(name)

    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "",
        parameters={"type": "object", "properties": properties, "required": required},
        function=function
    )

**装饰器**

`tool` 可以当作**装饰器**来用：在函数定义的上面写一行 `@tool`。

```python
@tool
def get_current_time(...):
    ...
```

它的效果和下面这样写完全一样：

```python
def get_current_time(...):
    ...
get_current_time = tool(get_current_time)
```

也就是说，定义完函数以后，`get_current_time` 这个名字指向的就不再是原来的函数，而是 `tool` 返回的 `Tool` 对象了。多亏了 `Tool` 的 `__call__`，我们仍然可以像调用函数一样调用它。

来看看改写以后的两个工具。定义的同时，打印出自动生成的参数说明，和上一章手写的比较一下：

In [ ]:
@tool
def get_current_time(timezone: Annotated[str, "IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'"]) -> str:
    """Get the current local date and time in a given IANA timezone."""
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


print(json.dumps(get_current_time.parameters, indent=2))
print(get_current_time("America/Toronto"))

`add_days` 的参数 `days` 有默认值 `1`，所以自动生成的说明里，`required` 只有 `date`，`days` 变成了可选的。

In [ ]:
@tool
def add_days(date: Annotated[str, "Start date in YYYY-MM-DD format"],
             days: Annotated[int, "Number of days to add, negative to subtract"] = 1) -> str:
    """Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday."""
    result = datetime.strptime(date, "%Y-%m-%d") + timedelta(days=days)
    return result.strftime("%Y-%m-%d (%A)")


print(json.dumps(add_days.parameters, indent=2))
print(add_days("2026-11-10", 100))

## 工具箱 ToolRegistry

工具多了以后，需要一个地方统一管理它们。`ToolRegistry` 就是学习助手的**工具箱**：

- `register`：把一个工具放进工具箱。如果名字和已有的工具重复，就报错，因为大模型是按名字调用工具的，名字必须唯一。
- `list`：列出所有工具，交给大模型。
- `run`：按名字找到工具并执行。上一章 `Agent` 里的 `run_tool` 搬到了这里，出错时同样返回错误信息，而不是让程序崩溃。

In [ ]:
class ToolRegistry:

    def __init__(self, tools: List[Tool] = None):
        self.tools = {}
        for t in tools or []:
            self.register(t)

    def register(self, t: Tool) -> Tool:
        if t.name in self.tools:
            raise ValueError(f"duplicate tool name: '{t.name}'")
        self.tools[t.name] = t
        return t

    def list(self) -> List[Tool]:
        return list(self.tools.values())

    def run(self, call: ToolCall) -> str:
        t = self.tools.get(call.name)
        if t is None:
            return f"Error: unknown tool '{call.name}'"
        try:
            return t.run(call.arguments)
        except Exception as e:
            return f"Error: {e}"  # let the model see the error and try to fix it

试一下工具箱：调用 `add_days` 时不传 `days`，会用默认值 1；调用一个不存在的工具，会得到错误信息。

In [ ]:
registry = ToolRegistry([get_current_time, add_days])
print([t.name for t in registry.list()])
print(registry.run(ToolCall("1", "add_days", {"date": "2026-11-10"})))
print(registry.run(ToolCall("2", "get_weather", {})))

有了工具箱，以后不管工具从哪里来，都可以放进同一个箱子：下一章来自远程服务器的工具、第 15 章查资料的工具，甚至第 22 章里另一个智能体，都会以工具的身份放进工具箱。`Agent` 只管从工具箱里拿工具，不用关心它们是怎么来的。

In [ ]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, *messages: Message):
        # add a whole turn at once, so it is summarized (at most) once per turn
        self.messages.extend(messages)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        # never separate a tool call from its result: keep from the last UserMessage onwards
        while split > 0 and not isinstance(self.messages[split], UserMessage):
            split -= 1
        if split == 0:
            return
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        note("summary", self.history())

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [message_from_dict(m) for m in data["messages"]]
        return memory

In [ ]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        pass

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ChatCompletionToolParam(type="function", function={
                "name": tool.name, "description": tool.description, "parameters": tool.parameters})
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                param = ChatCompletionAssistantMessageParam(role="assistant", content=msg.content)
                if msg.tool_calls:
                    param["tool_calls"] = [
                        ChatCompletionMessageToolCallParam(id=call.id, type="function", function={
                            "name": call.name, "arguments": json.dumps(call.arguments)})
                        for call in msg.tool_calls
                    ]
                schema.append(param)
            elif isinstance(msg, ToolResultMessage):
                # OpenAI: one "tool" message per tool result
                schema.append(ChatCompletionToolMessageParam(role="tool", tool_call_id=msg.tool_call_id, content=msg.result))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        tool_calls = [
            ToolCall(call.id, call.function.name, json.loads(call.function.arguments))
            for call in (message.tool_calls or [])
        ]
        return AssistantMessage(message.content or "", tool_calls)

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ToolParam(name=tool.name, description=tool.description, input_schema=tool.parameters)
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                content = [TextBlockParam(type="text", text=msg.content)] if msg.content else []
                content += [
                    ToolUseBlockParam(type="tool_use", id=call.id, name=call.name, input=call.arguments)
                    for call in msg.tool_calls
                ]
                schema.append(AssistantMessageParam(role="assistant", content=content))
            elif isinstance(msg, ToolResultMessage):
                # Claude: all tool results of one turn go into a single "user" message
                block = ToolResultBlockParam(type="tool_result", tool_use_id=msg.tool_call_id, content=msg.result)
                if schema and schema[-1]["role"] == "user" and isinstance(schema[-1]["content"], list):
                    schema[-1]["content"].append(block)
                else:
                    schema.append(UserMessageParam(role="user", content=[block]))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        tool_calls = [
            ToolCall(block.id, block.name, block.input)
            for block in response.content if block.type == "tool_use"
        ]
        return AssistantMessage(text, tool_calls)

## Agent 换上工具箱

`Agent` 的变化很小：创建时传入的不再是工具列表，而是一个工具箱 `registry`；执行工具时，调用 `self.registry.run(call)`。其他部分和上一章完全一样。

In [ ]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage, registry: ToolRegistry = None, max_steps: int = 5):
        self.provider = provider
        self.system = system
        self.registry = registry or ToolRegistry()
        self.max_steps = max_steps
        self.store = MemoryStore()
        self.memory = Memory(provider)

    def run(self, message: str, system: SystemMessage = None) -> AssistantMessage:
        system = system or self.system
        tools = self.registry.list()
        turn = [UserMessage(message)]

        response = self.provider.call(system, self.memory.messages + turn, tools)
        turn.append(response)

        step = 0
        while response.tool_calls and step < self.max_steps:
            step += 1
            for call in response.tool_calls:
                result = self.registry.run(call)
                note(f"tool[{step}]", f"{call} -> {result}")
                turn.append(ToolResultMessage(call.id, call.name, result))
            response = self.provider.call(system, self.memory.messages + turn, tools)
            turn.append(response)

        if response.tool_calls:
            response.tool_calls = []
            response.content = response.content or f"(stopped after {self.max_steps} steps)"

        self.memory.add(*turn)
        self.store.save(self.memory)
        return response

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    note(f"loaded memory #{memory.id}", memory.history())
                continue

            response = self.run(message, system)
            print(f"AI: {response.content}")

In [ ]:
provider = OpenAIProvider()
agent = Agent(provider, system_message, registry=registry)
agent.chat(["我想每 3 天复习一次，今天开始，第 4 次复习是哪天、星期几？"])
# or call agent.chat() without questions to type your own

## 写好工具的描述

工具的名字和描述，其实也是写给大模型的**提示词**。大模型能不能选对工具、填对参数，很大程度上取决于它们写得好不好：

| | 不好的写法 | 好的写法 |
|---|---|---|
| 名字 | `tool1` | `add_days` |
| 描述 | “处理日期” | “给一个日期加上（或减去）若干天，返回新的日期和星期几” |
| 参数说明 | “日期” | “起始日期，格式为 YYYY-MM-DD” |

好的描述说清楚三件事：**它能做什么、什么时候该用、参数该怎么填**。

> 很多智能体框架都有和 `@tool` 类似的装饰器。真实的项目里，通常会用 Pydantic 这样的库来生成更完整的参数说明，原理和我们这里写的是一样的。

## 动手试试

- 用 `@tool` 写一个新工具 `days_between(start_date, end_date)`，计算两个日期之间相差多少天，加入工具箱，然后问学习助手“离 12 月 20 日的考试还有几天？”。
- 把 `add_days` 的描述改得含糊一点，比如只写“处理日期”，再问同样的问题，看看大模型还会不会正确地使用它。
- `JSON_TYPES` 里，`float` 对应的是 `number`。用 `@tool` 写一个带小数参数的工具，比如 `average(scores: list, weight: float)`，打印它的 `parameters`，看看生成的说明是不是你预想的样子。

## 小结

- 手写 JSON Schema 和函数本身是重复的。用 `inspect` 读取函数的名字、文档字符串、参数、类型标注和默认值，就能**自动生成**工具说明。
- `Annotated` 可以给参数的类型附带说明文字。
- `@tool` 是一个**装饰器**：`@tool` 写在函数上面，等于 `函数 = tool(函数)`。
- `ToolRegistry` 是统一管理工具的**工具箱**，负责注册、列出和执行工具。`Agent` 只和工具箱打交道。
- 工具的名字和描述也是提示词，要写清楚它能做什么、什么时候用、参数怎么填。

到目前为止，学习助手的工具都是我们自己写的函数。下一章，我们让它连接到**别人提供的工具服务器**：通过 MCP 协议，直接用上远程服务器上现成的工具。